In [10]:
# Setup
!pip install uv -q
!git clone --depth 1 https://github.com/DataTalksClub/machine-learning-zoomcamp.git
%cd machine-learning-zoomcamp/cohorts/2026/homework/05-deployment

!uv sync --locked
!uv run python smoke_test.py
!sha256sum pipeline.bin

Cloning into 'machine-learning-zoomcamp'...
remote: Enumerating objects: 1913, done.
remote: Counting objects: 100% (1913/1913), done.
remote: Compressing objects: 100% (1803/1803), done.
remote: Total 1913 (delta 120), reused 1652 (delta 78), pack-reused 0 (from 0)
Receiving objects: 100% (1913/1913), 70.95 MiB | 29.21 MiB/s, done.
Resolving deltas: 100% (120/120), done.
/kaggle/working/machine-learning-zoomcamp/cohorts/2026/homework/05-deployment/machine-learning-zoomcamp/cohorts/2026/homework/05-deployment
Using CPython 3.11.15
Creating virtual environment at: .venv
Resolved 19 packages in 1ms
░░░░░░░░░░░░░░░░░░░░ [0/18] Installing wheels...                                warning: Failed to hardlink files; falling back to full copy. This may lead to degraded performance.
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this

In [11]:
# Q1
!uv --version

uv 0.13.5 (x86_64-unknown-linux-gnu)


In [12]:
# Q2
!grep -A2 'name = "scikit-learn"' uv.lock

    { name = "scikit-learn" },
    { name = "uvicorn" },
]
--
    { name = "scikit-learn", specifier = "==1.7.2" },
    { name = "uvicorn", specifier = "==0.38.0" },
]
--
name = "scikit-learn"
version = "1.7.2"
source = { registry = "https://pypi.org/simple" }


In [14]:
# Q3
code = '''
import pickle

with open("pipeline.bin", "rb") as f:
    model = pickle.load(f)

lead = {
    "lead_source": "paid_ads",
    "industry": "technology",
    "employment_status": "employed",
    "location": "north_america",
    "number_of_courses_viewed": 2,
    "annual_income": 79276.0,
    "interaction_count": 4,
    "lead_score": 0.41
}

print(round(model.predict_proba([lead])[0, 1], 3))
'''

with open("q3.py", "w") as f:
    f.write(code)

!uv run python q3.py

0.533


In [17]:
# Q4 - start the API in the background
import subprocess, time

server = subprocess.Popen(
    "uv run uvicorn predict:app --host 0.0.0.0 --port 9696 > api.log 2>&1",
    shell=True
)
time.sleep(8)

In [18]:
# Q4 - send the request
!curl -s http://localhost:9696/health
!curl -s http://localhost:9696/predict -H 'Content-Type: application/json' -d '{"lead_source": "organic_search", "industry": "technology", "employment_status": "employed", "location": "europe", "number_of_courses_viewed": 4, "annual_income": 80304.0, "interaction_count": 7, "lead_score": 0.74}'

{"status":"ok","model_sha256":"1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1","feature_defaults_sha256":"1694f842b908fac9f53c0cc19be6d91f934befc63af23cc32fefafde69acd0cd"}{"conversion_probability":0.769799,"conversion":true}

In [19]:
# Q5
!grep FROM Dockerfile

FROM python:3.11.15-slim-bookworm@sha256:d29f48a31a8b408ed19272ca1e7b10ebae13b240a27e862d3d4217c528e2e0c3


In [20]:
# Q6 - client test (Docker can't run on Kaggle)
!uv run python q6_test.py

{'conversion_probability': 0.769799, 'conversion': True}
